## exploring the name list published by the national office of statistics 
https://www.ons.gov.uk/peoplepopulationandcommunity/birthsdeathsandmarriages/livebirths/datasets/babynamesinenglandandwalesfrom1996


In [3]:
# checking which sheets the people's names excel file has 

import pandas as pd

sheets = pd.read_excel("/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/raw/reference/babynames1996to2025.xlsx", sheet_name=None)
print("Sheets in the Excel file:")
for sheet_name in sheets.keys():
    print(sheet_name)

Sheets in the Excel file:
Cover_sheet
Contents
Notes
Table_1
Table_2


### read in boys and girls names and concat to one name list

In [ ]:
df_raw_boy = pd.read_excel("/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/raw/reference/babynames1996to2025.xlsx", sheet_name="Table_2")
boy_names = df_raw_boy.iloc[4:].copy()
boy_names.columns = df_raw_boy.iloc[3]
boy_names.reset_index(drop=True, inplace=True)
boy_names = boy_names[['Name']]

df_raw_girl = pd.read_excel("/mnt/xdrive/spandh_bessemer/SPandhAndHealthcare/Fritz/dementia-speech/data/raw/reference/babynames1996to2025.xlsx", sheet_name="Table_1")
girl_names = df_raw_girl.iloc[4:].copy()
girl_names.columns = df_raw_girl.iloc[3]
girl_names.reset_index(drop=True, inplace=True)
girl_names = girl_names[['Name']]

all_names = pd.concat([boy_names, girl_names], ignore_index=True)
all_names

### see which names in the list have a synset in wordnet 
- the idea: if they have a synset, they are actually a content word and not a name 
- problem: some these names have a synset because they are linked to famous people or other instance

In [19]:
from nltk.corpus import wordnet as wn

p_names = all_names[all_names['Name'].str.lower().str.startswith('p')].copy()
p_names['is_real_word'] = p_names['Name'].apply(lambda w: len(wn.synsets(w.lower())) > 0)

collisions = sorted(p_names[p_names['is_real_word']]['Name'].str.lower().unique())
print(len(collisions), 'collision candidates')
print(collisions)

108 collision candidates
['pa', 'pace', 'pacha', 'paddy', 'pagan', 'page', 'paige', 'paisley', 'pal', 'palmer', 'pandora', 'pansy', 'papa', 'paradise', 'paras', 'paris', 'parish', 'pariss', 'parker', 'pars', 'parvati', 'pascal', 'pascha', 'pasha', 'passion', 'pat', 'patience', 'patrick', 'patsy', 'paul', 'pauline', 'pauls', 'pavan', 'pax', 'paxton', 'peace', 'peach', 'peaches', 'pearl', 'pearly', 'pebbles', 'peer', 'pei', 'pelham', 'penelope', 'penn', 'penni', 'penny', 'peony', 'pepper', 'percy', 'peregrine', 'peri', 'perry', 'persephone', 'perseus', 'persia', 'pesach', 'petal', 'peter', 'pharaoh', 'phi', 'philemon', 'philip', 'phoebe', 'phoenix', 'pia', 'pierce', 'pierre', 'piers', 'pilar', 'pinches', 'pious', 'pip', 'piper', 'pippin', 'pixie', 'plum', 'poem', 'poet', 'pol', 'poppy', 'porter', 'poseidon', 'posy', 'prairie', 'praise', 'precious', 'prentice', 'presley', 'pretty', 'prevail', 'priam', 'prim', 'primo', 'primrose', 'prince', 'princess', 'princeton', 'promise', 'prosper', 'p

### refine collision problem 

- Collision problem: many names double as ordinary English words (penny, pearl, prince, peace...). Naively flagging any gazetteer match as a proper noun would wrongly exclude legitimate correct responses. Fix: only classify a gazetteer match as a proper noun if the word has no ordinary-word sense in WordNet — i.e. WordNet-first, gazetteer as fallback.
- Refinement: a plain WordNet lookup over-corrects, because WordNet also encodes specific famous individuals (paul→Paul the Apostle, patrick→Saint Patrick) as "instance" senses distinct from ordinary "class" senses. Filtering out instance-only senses (has_ordinary_sense, via synset.instance_hypernyms()) rescued 22 names that would otherwise have been wrongly treated as ordinary words.
- Manual override list (FORCE_NAME_EXCEPTIONS): a small hand-curated set for residual cases the automatic rule still gets wrong — e.g. pauline, which WordNet defines as a genuine (but obscure) adjective ("Pauline theology"), not an instance, so the automatic fix doesn't catch it. Grown by manual review as more collisions are found.

In [22]:
from nltk.corpus import wordnet as wn

def has_ordinary_sense(word: str) -> bool:
    """True if the word has at least one WordNet sense that is NOT
    a specific named individual (e.g. 'paul' the apostle) — i.e. a
    genuine common-word meaning."""
    for s in wn.synsets(word.lower().strip()):
        if not s.instance_hypernyms():
            return True
    return False

p_names = all_names[all_names['Name'].str.lower().str.startswith('p')].copy()
p_names['is_real_word'] = p_names['Name'].apply(lambda w: has_ordinary_sense(w))

collisions = sorted(p_names[p_names['is_real_word']]['Name'].str.lower().unique())
print(len(collisions), 'collision candidates')
print(collisions)

86 collision candidates
['pa', 'pace', 'pacha', 'paddy', 'pagan', 'page', 'paisley', 'pal', 'pansy', 'papa', 'paradise', 'paras', 'paris', 'parish', 'pariss', 'pars', 'pascal', 'pascha', 'pasha', 'passion', 'pat', 'patience', 'patsy', 'pauline', 'pavan', 'pax', 'peace', 'peach', 'peaches', 'pearl', 'pearly', 'pebbles', 'peer', 'pelham', 'penelope', 'penni', 'penny', 'peony', 'pepper', 'peregrine', 'peri', 'perry', 'pesach', 'petal', 'peter', 'pharaoh', 'phi', 'phoebe', 'phoenix', 'pia', 'pierce', 'piers', 'pilar', 'pinches', 'pious', 'pip', 'piper', 'pippin', 'pixie', 'plum', 'poem', 'poet', 'pol', 'poppy', 'porter', 'posy', 'prairie', 'praise', 'precious', 'prentice', 'pretty', 'prevail', 'prim', 'primo', 'primrose', 'prince', 'princess', 'promise', 'prosper', 'prudence', 'prunella', 'prys', 'psalm', 'psalms', 'ptolemy', 'purity']


In [ ]:
FORCE_NAME_EXCEPTIONS = {
    "paddy",
    "pauline",
    "penelope",
    "peter",
    "phoebe"
}

In [21]:
p_names

3,Name,is_real_word
12972,Pa,True
12973,Paa,False
12974,Paapa,False
12975,Paarth,False
12976,Paavan,False
...,...,...
35892,Purity,True
35893,Purva,False
35894,Purvi,False
35895,Puteri,False


In [24]:
import pycountry 

COUNTRY_SET = set(c.name.lower() for c in pycountry.countries)
COUNTRY_SET

{'afghanistan',
 'albania',
 'algeria',
 'american samoa',
 'andorra',
 'angola',
 'anguilla',
 'antarctica',
 'antigua and barbuda',
 'argentina',
 'armenia',
 'aruba',
 'australia',
 'austria',
 'azerbaijan',
 'bahamas',
 'bahrain',
 'bangladesh',
 'barbados',
 'belarus',
 'belgium',
 'belize',
 'benin',
 'bermuda',
 'bhutan',
 'bolivia, plurinational state of',
 'bonaire, sint eustatius and saba',
 'bosnia and herzegovina',
 'botswana',
 'bouvet island',
 'brazil',
 'british indian ocean territory',
 'brunei darussalam',
 'bulgaria',
 'burkina faso',
 'burundi',
 'cabo verde',
 'cambodia',
 'cameroon',
 'canada',
 'cayman islands',
 'central african republic',
 'chad',
 'chile',
 'china',
 'christmas island',
 'cocos (keeling) islands',
 'colombia',
 'comoros',
 'congo',
 'congo, the democratic republic of the',
 'cook islands',
 'costa rica',
 'croatia',
 'cuba',
 'curaçao',
 'cyprus',
 'czechia',
 "côte d'ivoire",
 'denmark',
 'djibouti',
 'dominica',
 'dominican republic',
 'ecua

In [38]:
wn.synonyms('UK')

[['Britain',
  'Great_Britain',
  'U.K.',
  'United_Kingdom',
  'United_Kingdom_of_Great_Britain_and_Northern_Ireland']]